## Cell 1 — Imports & config  (unchanged)

In [ ]:
import requests, urllib3
import pytz

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

file_date  = "2026-06-02"
url        = "<YOUR_ELASTICSEARCH_URL>/_search"
user       = "<USERNAME>"
psd        = "<PASSWORD>"
headers    = {"Content-Type": "application/json"}
all_scopes = ['SWCC', 'TDCC', 'PBCC', 'PTCC']

kolkata_tz      = pytz.timezone("Asia/Kolkata")
INTERVAL_MIN    = 10
total_intervals = (24 * 60) // INTERVAL_MIN   # 144

## Cell 2 — Fetch loop  (unchanged — builds `all_hits`)

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

WINDOW_MIN = 30  # minutes per window

# Group all_hits by 30-min window using @timestamp
windows = {}
for record in all_hits:
    fields  = record.get("fields", {})
    ts_raw  = fields.get("@timestamp", [None])
    ts_str  = ts_raw[0] if isinstance(ts_raw, list) else ts_raw
    if ts_str:
        dt      = datetime.fromisoformat(ts_str[:16])
        win_key = (dt.hour * 60 + dt.minute) // WINDOW_MIN   # 0–47
    else:
        win_key = -1
    windows.setdefault(win_key, []).append(record)

total_windows = 24 * 60 // WINDOW_MIN   # 48

# Build one DF per window, union them — driver serialises ~19k records at a time (not 9 lakh).
# Union is lazy in Spark, so the final df is processed in chunks — no OOM.
# df is available after the loop so display(df) works normally.
df = None

for win_key in sorted(windows.keys()):
    records   = windows[win_key]
    start_min = win_key * WINDOW_MIN
    h, m      = divmod(start_min, 60)
    print(f"Window {win_key + 1}/{total_windows}  [{h:02d}:{m:02d} – {h:02d}:{m + WINDOW_MIN - 1:02d}]:  {len(records):,} records")

    flattened_records = []
    for record in records:
        flat = {}
        for key, value in record["fields"].items():
            flat[key] = value[0] if isinstance(value, list) and len(value) > 0 else value
        flattened_records.append(flat)

    window_df = spark.createDataFrame(flattened_records)
    df = window_df if df is None else df.union(window_df)

    del window_df, flattened_records, records
    windows[win_key] = None

print(f"total columns: {len(df.columns)}")

## Cell 3 — DataFrame creation  (**CHANGED** — 30-minute windows)

**Before:** `spark.createDataFrame(all_flattened)` — loads all ~9 lakh records at once → driver OOM  
**After:** groups `all_hits` by 30-min window using `@timestamp`, creates one small DataFrame per window, then frees memory

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

WINDOW_MIN = 30  # size of each window in minutes

# ── Group all_hits by 30-min window using @timestamp ─────────────────────────
windows = {}
for record in all_hits:
    fields  = record.get("fields", {})
    ts_raw  = fields.get("@timestamp", [None])
    ts_str  = ts_raw[0] if isinstance(ts_raw, list) else ts_raw
    if ts_str:
        dt      = datetime.fromisoformat(ts_str[:16])          # "2026-06-02T00:09"
        win_key = (dt.hour * 60 + dt.minute) // WINDOW_MIN     # 0 – 47
    else:
        win_key = -1
    windows.setdefault(win_key, []).append(record)

total_windows = 24 * 60 // WINDOW_MIN   # 48

# ── Process one 30-min window at a time ──────────────────────────────────────
for win_key in sorted(windows.keys()):
    records   = windows[win_key]
    start_min = win_key * WINDOW_MIN
    h, m      = divmod(start_min, 60)
    print(f"\nWindow {win_key + 1}/{total_windows}  [{h:02d}:{m:02d} – {h:02d}:{m + WINDOW_MIN - 1:02d}]:  {len(records):,} records")

    # flatten fields (same logic as before)
    flattened_records = []
    for record in records:
        flat = {}
        for key, value in record["fields"].items():
            flat[key] = value[0] if isinstance(value, list) and len(value) > 0 else value
        flattened_records.append(flat)

    df = spark.createDataFrame(flattened_records)
    print(f"  total columns: {len(df.columns)}")

    # ── add your transformations / writes here ────────────────────────────────
    # e.g. df.write.mode("append").parquet(f"/mnt/output/{file_date}/window_{win_key+1:02d}")
    # ─────────────────────────────────────────────────────────────────────────

    # free this window's memory before moving to the next
    del df, flattened_records, records
    windows[win_key] = None

## Cell 4 — Extract `pan_number` & `crm_lead_number` from the `message` column

The `message` column holds JSON strings with **several different schemas** (SYSTEM_DATA / DOMAIN_DATA / …).
The PAN and CRM-lead values live at different nesting depths and under different key names:

| field | keys seen in data | example |
|-------|-------------------|---------|
| PAN | `"pan"`, `"panNumber"` (also inside `panDetails`) | `ABLPZ9835G`, `BEZPC5075H` |
| CRM lead | `"crmLeadNumber"` (inside `crmDetails`) | `845052814` |

Because the schema is inconsistent (and some rows are malformed), we extract straight from the raw
string with `regexp_extract` instead of parsing JSON — this is agnostic to nesting depth and schema variant.
PAN's strict format `[A-Z]{5}[0-9]{4}[A-Z]` makes it safe to match.

In [ ]:
from pyspark.sql import functions as F

# ── Regex patterns ──────────────────────────────────────────────────────────
# PAN  : key is "pan" or "panNumber"; value is a valid PAN  [A-Z]{5}[0-9]{4}[A-Z]
#        (anchoring on the key avoids matching "panValid" / "panDetails" / "panOrForm60")
pan_pattern = r'"pan(?:Number)?"\s*:\s*"([A-Z]{5}[0-9]{4}[A-Z])"'

# CRM  : key is "crmLeadNumber" (case-insensitive on the 'l'); value is digits, quoted or not
crm_pattern = r'"crm[Ll]eadNumber"\s*:\s*"?(\d+)"?'

# ── Add the two new columns ─────────────────────────────────────────────────
raw_df = (
    raw_df
    .withColumn("pan_number",      F.regexp_extract(F.col("message"), pan_pattern, 1))
    .withColumn("crm_lead_number", F.regexp_extract(F.col("message"), crm_pattern, 1))
)

# regexp_extract returns "" when there is no match -> convert to NULL so missing values are explicit
raw_df = (
    raw_df
    .withColumn("pan_number",
                F.when(F.col("pan_number") == "", None).otherwise(F.col("pan_number")))
    .withColumn("crm_lead_number",
                F.when(F.col("crm_lead_number") == "", None).otherwise(F.col("crm_lead_number")))
)

display(raw_df.select("pan_number", "crm_lead_number", "message"))